# Data and Methods

To study phytoplankton phenology along the Northeast US Continental Shelf (NES), we are using the OCCCI v6.0 dataset, which was released in 2022 (ESA) and is updated quarterly to have up-to-date products. Products produced by OCCCI are available at a 4 km and 1 km spatial resolution. Their temporal coverage includes daily, 5-day, 8-day, monthly, and annual composites for the 4 km resolution products and daily, 8-day, monthly, and annual composites for the 1 km resolution products (ESA). We are using their chlorophyll-a concentration products at the 4 km annual and daily resolutions. To create the climatology and daily datasets used for this project, we created a subset of the data that included chlorophyll-a data from the NES region, bounded geographically by $34.02083^oN$, $46.97917^oN$, $61.02083^oW$, and $78.97917^oW$. These datasets were created by the National Oceanic and Atmospheric Administration (NOAA) National Marine Fisheries Service (NMFS) Northeast Fisheries Science Center (NEFSC). 

To create the regional climatology for comparison, we used the 4 km annual products for each year from 1998 - 2020. These data were averaged(?) together to create a climatological file that has been used to set threshold values for our detection methods. The data has been averaged for each pixel, so we have a climatological median for each 4 km pixel in the NES. To compute the bloom metrics, we used the daily 4 km products and applied a rolling 8 day weighted mean to fill missing data due to cloud interference or sensor interference and smooth the data for analysis. This was done for each day from September 4, 1997 to September 30, 2025 to create our time series data. The daily data is the main data for the project and is compared to the annual climatology for the NES region and ECUs, allowing for analysis of long-term bloom trends.

To test bloom detection methods and analyze bloom metrics, we used Python (3.13). 

#### Smoothing of chlorophyll data
For this method and further analysis, we further smoothed the data to remove any daily fluctuations that would create noise for the analysis. We tested two smoothing techniques, the LOWESS technique and the Savitsky-Golay technique. We decided to use the Savitsky_Golay technique because it more accurately captured bloom magnitude, correctly identifying more bloom events for shorter duration events than the LOWESS function. The processes are laid out below.

Lowess smoothing uses the points surrounding a point (the fraction of points surrounding a data point) to estimate the value of that point (statsmodels webpage). This helps smooth out the naturally noisy chlorophyll-a data. Since the data has been previously smoothed using a rolling 8-day mean, only light smoothing is necessary to remove the jagged nature of bloom peaks. We tested a variety of fractions that smoothed from 5 - 365 days for the full time series and found that the best fit was 0.00117, which is the equivalent of a 12-day window on the full time series. This value smooths out neglible chlorophyll-a concentration spikes, while still accurately modelling the major chl-a events. For verfication of the fractional value chosen, a time series of one year of data was plotted alongside its smoothed value. This process was repeated with a variety of years in the time series and in all of the regions of the NES. A fraction value of 0.035 smooths over approximately 12.77 days in a year long time series. To keep this level of smoothing consistent, a fractional value of 0.00117 was chosen to smooth the entire time series. This provides the same smoothing resolution as the 0.035 smoother on the yearly data. While this captured major bloom events well, it cut off a lot of bloom magnitude, causing some smaller duration blooms to be missed in bloom analysis.

The Savitsky-Golay (SG) method fits a mathematical equation to the data within a specified window. For testing, we tested windows of 7, 15, 21, and 31 days. We found that a 7 day window did not smooth the data enough for our analysis but that the 15 day window captured most blooms above the threshold while still smoothing small daily fluctuations. For this reason, we chose this method with a 15 day window and a polynomial fit of 3. The polynomial fit does not change the points or the graphed data, but it does change the ease of calculating the rate of change. A cubic function was easier to get the rate of change of, therefore it was chosen over a quadratic function.

### Lowess Smoothing

In [ ]:
import itertools
from sklearn.metrics import mean_squared_error, r2_score
frac_values = [0.00097, 0.00107, 0.00117, 0.00127,0.00136, 0.00145] 
it_values = [3, 5]
region_data = [MABS,MABN,GB,GOMW,GOME]
region_titles = ['MABS','MABN','GB','GOMW','GOME']
for x in range(5):
    data = region_data[x]
    region_title = region_titles[x]
    time = data.time.astype('int64')
    median=data.to_dataframe().reset_index()
    median = median["CHL_median"].values

    # itertools.product creates every possible combination of your lists
    for frac, it in itertools.product(frac_values, it_values):
        print(f"Running LOWESS: frac={frac}, it={it}...")
        
        # 1. Create a boolean mask of valid (non-NaN) indices for both arrays
        valid_mask = ~np.isnan(median)
        
        #2. Slice both arrays to only include days with valid numbers
        clean_median = median[valid_mask]
        clean_time = time[valid_mask]

        # Run the LOWESS function
        # return_sorted=False ensures the output matches our input array exactly
        smoothed_output = sm.nonparametric.smoothers_lowess.lowess(
            endog=clean_median, 
            exog=clean_time, 
            frac=frac, 
            it=it, 
            return_sorted=False
        )

        smoothed_full = np.full(len(data['time']), np.nan)
        smoothed_full[valid_mask] = smoothed_output

        variable_name = f"smoothed_lowess_frac_{frac}_int_{it}"
        data[variable_name] = (['time'],smoothed_full)

    data.to_zarr(rf'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\{region_title}_smoother_tests.zarr',mode='w')
    print(f"Successfully saved {region_title} zarr file with all testing variables")

### Savitsky Golay Smoothing

In [ ]:
from scipy.signal import savgol_filter
region_data = [MABN,GB,GOMW,GOME]
region_titles = ['MABN','GB','GOMW','GOME']
for x in range(4):
    data = region_data[x]
    region_title = region_titles[x]
    time = data.time.astype('int64')
    median=data.to_dataframe().reset_index()
    median = median["CHL_median"].values
    mask = ~np.isnan(median)
    chl_interp = pd.Series(median).interpolate(method='linear').bfill().ffill().values
    window_lengths = [7,15]
    poly_order = [3]

    for window,poly in itertools.product(window_lengths,poly_order):
        sg_smoothed = savgol_filter(chl_interp,window_length=window,polyorder=poly)
        sg_smoothed_full = np.copy(sg_smoothed)
        sg_smoothed_full[~mask]=np.nan
        variable_name = f"smoothed_sg_win_{window}_poly_{poly}"
        data[variable_name]=(['time'],sg_smoothed_full)
    data.to_zarr(rf'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\{region_title}_smoother_tests.zarr',mode='a')
    print("Success!")

### Plotting the Smoothers for Analysis

In [ ]:
time_series = [1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024]
region_titles = ["MAB South","MAB North","Georges Bank","Gulf of Maine West","Gulf of Maine East"]
region_acro = ["MABS","MABN","GB","GOMW","GOME"]
for x in range(5):
    region_title = region_titles[x]
    smoother = xr.open_zarr(rf'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\{region_acro[x]}_smoother_tests.zarr')
    for year in time_series:  
        start_year = year-1
        end_year = year+1
        fig, ax = plt.subplots(figsize=(16,8))
        smoother['CHL_median'].plot(ax=ax, c='darkorange',label="Raw data",alpha=0.5)
        smoother['smoothed_lowess_frac_0.00117_int_3'].plot(ax=ax, c='blue',label="Lowess Frac: 12 days, int: 3",alpha=0.8)
        #smoother['smoothed_lowess_frac_0.00145_int_3'].plot(ax=ax, c='green',label="Frac: 15 days, int: 3",alpha=0.8)
        smoother['smoothed_sg_win_7_poly_3'].plot(ax=ax, c='red',label="SG Window = 7 days",alpha=0.8)
        smoother['smoothed_sg_win_15_poly_3'].plot(ax=ax, c='green',label="SG Window = 15 days",alpha=0.8)
        plt.axhline(clim_med[x][0],c="purple",label="Climatological median",alpha=0.3)
        plt.axhline(clim_10[x][0],c="red",label="10% Threshold",alpha=0.3)
        start = np.datetime64(f'{str(start_year)}-01-01','D').astype(int)
        end = np.datetime64(f'{str(end_year)}-01-01','D').astype(int)
        plt.xlim(start,end)
        plt.xlabel("Date")
        plt.ylabel("CHL-a Concentrations ($mg/m^3$)")
        plt.legend(fontsize=8)
        plt.title(f"Smoothers for {region_title} from {start_year}-{end_year}")
        filename = f"{region_acro[x]}_smoothers_{str(start_year)}_{str(end_year)}"
        plt.savefig(rf'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Smoother_testing\{filename}')
        plt.close()

Acknowledgement of OCCCI data
<br>"The European Space Agency (ESA) Climate Change Initiative (CCI) Ocean Colour data are developed and produced through funding and resources from ESA, Copernicus Climate Change Service (C3S) and the NERC EO Data Analysis and AI Service (NEODAAS)."

### Citations

Sathyendranath, S, Brewin, RJW, Brockmann, C, Brotas, V, Calton, B, Chuprin, A, Cipollini, P, Couto, AB, Dingle, J, Doerffer, R, Donlon, C, Dowell, M, Farman, A, Grant, M, Groom, S, Horseman, A, Jackson, T, Krasemann, H, Lavender, S, Martinez-Vicente, V, Mazeran, C, Mélin, F, Moore, TS, Müller, D, Regner, P, Roy, S, Steele, CJ, Steinmetz, F, Swinton, J, Taberner, M, Thompson, A, Valente, A, Zühlke, M, Brando, VE, Feng, H, Feldman, G, Franz, BA, Frouin, R, Gould, Jr., RW, Hooker, SB, Kahru, M, Kratzer, S, Mitchell, BG, Muller-Karger, F, Sosik, HM, Voss, KJ, Werdell, J, and Platt, T (2019) An ocean-colour time series for use in climate studies: the experience of the Ocean-Colour Climate Change Initiative (OC-CCI). Sensors: 19, 4285. doi:10.3390/s19194285
<br>Sathyendranath, S.; Jackson, T.; Brockmann, C.; Brotas, V.; Calton, B.; Chuprin, A.; Clements, O.; Cipollini, P.; Danne, O.; Dingle, J.; Donlon, C.; Grant, M.; Groom, S.; Krasemann, H.; Lavender, S.; Mazeran, C.; Mélin, F.; Müller, D.; Steinmetz, F.; Valente, A.; Zühlke, M.; Feldman, G.; Franz, B.; Frouin, R.; Werdell, J.; Platt, T. (2023): ESA Ocean Colour Climate Change Initiative (Ocean_Colour_cci): Monthly climatology of global ocean colour data products at 4km resolution, Version 6.0. NERC EDS Centre for Environmental Data Analysis, date of citation. https://catalogue.ceda.ac.uk/uuid/690fdf8f229c4d04a2aa68de67beb733/
